# Preference Parsing

## Objective
Turn natural-language recommendation requests into a compact, inspectable request contract that downstream retrieval, reranking, grounding, and response generation can consume directly. The goal is not to make retrieval smarter here, but to represent user intent clearly enough that later stages can honor constraints, preferences, and clarification needs without repeatedly guessing from raw text.

## Inputs
- `../data/baseline_retrieval/candidate_pool.parquet`
- `../data/baseline_retrieval/index_metadata.json`
- `../data/curated/scenario_suite.json`
- the retrieval assumptions established in `02_baseline_retrieval.ipynb`

## Outputs
- a stable request schema
- a deterministic parser backed by shared helpers in `functions/`
- parsed scenario examples and pipeline handoff objects


## Why A Parser Still Matters

Even with a stronger retrieval baseline, a conversational request is not just a bag of words. Users mix together item types, use cases, exclusions, price expectations, and references to earlier context. The parser is where that mixed input becomes a small internal contract that later stages can inspect rather than infer implicitly.


In [9]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

PROJECT_ROOT_CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
for candidate in PROJECT_ROOT_CANDIDATES:
    if (candidate / "functions").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the project root containing functions/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from functions.io import load_curated_scenarios, load_retrieval_artifacts, materialize_scenarios
from functions.parsing import (
    REQUEST_SCHEMA_TEMPLATE,
    build_parser_catalog,
    build_pipeline_handoff,
    flatten_parsed_output,
    parse_user_request,
)

artifacts = load_retrieval_artifacts(load_encoder_model=False)
candidate_pool = artifacts["candidate_pool"]
index_metadata = artifacts["index_metadata"]
parser_catalog = build_parser_catalog(candidate_pool)
scenario_suite = load_curated_scenarios()
demo_scenarios = materialize_scenarios(scenario_suite["demo_scenarios"], candidate_pool)

baseline_summary = pd.DataFrame(
    [
        {
            "baseline_name": index_metadata.get("baseline_name"),
            "candidate_items": len(candidate_pool),
            "encoder_model": index_metadata.get("encoder_model"),
            "embedding_dim": index_metadata.get("embedding_dim"),
            "faiss_index_type": index_metadata.get("faiss_index_type"),
            "hybrid_pool_size": index_metadata.get("hybrid_pool_size"),
            "price_policy": index_metadata.get("candidate_pool_strategy", {}).get("price_policy"),
        }
    ]
)

scenario_catalog = pd.DataFrame(
    [
        {
            "label": scenario["label"],
            "title": scenario["title"],
            "query": scenario["query"],
            "has_reference_context": scenario["reference_item_context"] is not None,
        }
        for scenario in demo_scenarios
    ]
)

display(baseline_summary)
display(scenario_catalog)

,baseline_name,candidate_items,encoder_model,embedding_dim,faiss_index_type,hybrid_pool_size,price_policy
0,metadata_hybrid_v3_300k_source_aware,300000,BAAI/bge-small-en-v1.5,384,IndexFlatIP,500,Exclude items with null price from the retrievable pool because they were likely not purchasable when the snapshot was collected.


,label,title,query,has_reference_context
0,budget_headphones_avoid_beats,Highly rated headphones under budget,"Recommend wireless bluetooth headphones under 80 dollars that are highly rated, but avoid Beats.",False
1,reference_cheaper_headphones,Cheaper alternative to a reference item,I want something like this item but cheaper.,True
2,induction_frying_pan,Cookware request with a use-case constraint,Show me a nonstick frying pan for induction cooking.,False
3,dumbbells_home_gym,Sports query with a clear equipment type,Recommend adjustable dumbbells for home gym workouts.,False
4,lightweight_programming_laptop,Use-case request with portability preference,Show me a lightweight laptop for programming.,False
5,gift_ambiguity,Underspecified gifting request,"Find me something good for gifting, not too expensive.",False


## Contract Design

The schema stays deliberately small. Every field earns its place by answering a practical downstream question: how should we retrieve candidates, which constraints are hard, which preferences are soft, which claims later need evidence, and when should the system ask for clarification instead of pretending it fully understood the request.


In [10]:
schema_fields = pd.DataFrame(
    [
        {
            "field": "original_query",
            "type": "string",
            "why_it_exists": "Preserves the raw request for traceability and fallback behavior.",
        },
        {"field": "user_intent", "type": "string enum", "why_it_exists": "Chooses the broad pipeline path."},
        {
            "field": "domain_hint",
            "type": "nullable string",
            "why_it_exists": "Steers retrieval toward the right source when the request makes that possible.",
        },
        {
            "field": "hard_constraints",
            "type": "nested object",
            "why_it_exists": "Separates enforceable conditions from softer preferences.",
        },
        {
            "field": "soft_preferences",
            "type": "list[string]",
            "why_it_exists": "Preserves ranking preferences that should not behave like hard filters.",
        },
        {
            "field": "reference_items",
            "type": "list[object]",
            "why_it_exists": "Supports context-aware similarity retrieval.",
        },
        {
            "field": "excluded_brands / excluded_terms",
            "type": "list[string]",
            "why_it_exists": "Captures negative constraints that later stages can enforce explicitly.",
        },
        {
            "field": "grounding_needs",
            "type": "list[string]",
            "why_it_exists": "Marks which later claims deserve supporting evidence.",
        },
        {
            "field": "clarification_needed / clarification_questions",
            "type": "bool + list[string]",
            "why_it_exists": "Keeps ambiguity explicit instead of silently overcommitting.",
        },
    ]
)

print(json.dumps(REQUEST_SCHEMA_TEMPLATE, indent=2))
display(schema_fields)

{
  "original_query": null,
  "user_intent": null,
  "domain_hint": null,
  "hard_constraints": {
    "max_price": null,
    "min_rating": null,
    "must_include_terms": [],
    "use_case": null,
    "cheaper_than_reference": false,
    "same_source_as_reference": false
  },
  "soft_preferences": [],
  "reference_items": [],
  "excluded_brands": [],
  "excluded_terms": [],
  "grounding_needs": [],
  "clarification_needed": false,
  "clarification_questions": []
}


,field,type,why_it_exists
0,original_query,string,Preserves the raw request for traceability and fallback behavior.
1,user_intent,string enum,Chooses the broad pipeline path.
2,domain_hint,nullable string,Steers retrieval toward the right source when the request makes that possible.
3,hard_constraints,nested object,Separates enforceable conditions from softer preferences.
4,soft_preferences,list[string],Preserves ranking preferences that should not behave like hard filters.
5,reference_items,list[object],Supports context-aware similarity retrieval.
6,excluded_brands / excluded_terms,list[string],Captures negative constraints that later stages can enforce explicitly.
7,grounding_needs,list[string],Marks which later claims deserve supporting evidence.
8,clarification_needed / clarification_questions,bool + list[string],Keeps ambiguity explicit instead of silently overcommitting.


## Curated Scenario Parsing

The parser is easier to judge on a small set of representative requests than on isolated toy phrases. The scenarios below cover the main kinds of input this project needs to handle: explicit constraints, reference-based refinement, use-case hints, exclusions, and ambiguity that should trigger clarification.

The goal is not to claim full language understanding. The goal is to check whether the parser produces a stable internal contract that later stages can actually use for retrieval, reranking, grounding, and response generation.

In [11]:
parsed_examples = []
for scenario in demo_scenarios:
    parsed = parse_user_request(
        scenario["query"],
        parser_catalog,
        reference_item_context=scenario["reference_item_context"],
    )
    parsed_examples.append({**scenario, "parsed": parsed})

inspection_df = pd.DataFrame(
    [flatten_parsed_output(example["label"], example["parsed"]) for example in parsed_examples]
)

display(inspection_df)

for example in parsed_examples:
    print("=" * 100)
    print(example["label"])
    print(example["query"])
    print(json.dumps(example["parsed"], indent=2, ensure_ascii=False))

,label,user_intent,domain_hint,must_include_terms,use_case,max_price,min_rating,cheaper_than_reference,excluded_brands,soft_preferences,grounding_needs,reference_title,clarification_needed
0,budget_headphones_avoid_beats,constrained_search,electronics,headphones,NaN,80.00,None,False,Beats,highly_rated,"price, rating, brand_constraint",NaN,False
1,reference_cheaper_headphones,similar_item_refinement,electronics,,NaN,21.99,None,True,,budget_friendly,"price, reference_comparison",TOZO T10 Bluetooth 5.3 Wireless Earbuds with Wireless Charging Case IPX8 Waterproof Stereo Headphones in Ear Built in Mic Headset Premium Sound with Deep Bass for Sport White (2022 Upgraded),False
2,induction_frying_pan,constrained_search,home_and_kitchen,frying pan,induction cooking,NaN,None,False,,,use_case_fit,NaN,False
3,dumbbells_home_gym,constrained_search,sports_and_outdoors,"dumbbells, dumbbell",home gym workouts,NaN,None,False,,,use_case_fit,NaN,False
4,lightweight_programming_laptop,constrained_search,electronics,laptop,programming,NaN,None,False,,lightweight,"use_case_fit, portability",NaN,False
5,gift_ambiguity,gift_search,NaN,,gifting,NaN,None,False,,"budget_friendly, giftable","use_case_fit, giftability",NaN,True


budget_headphones_avoid_beats
Recommend wireless bluetooth headphones under 80 dollars that are highly rated, but avoid Beats.
{
  "original_query": "Recommend wireless bluetooth headphones under 80 dollars that are highly rated, but avoid Beats.",
  "user_intent": "constrained_search",
  "domain_hint": "electronics",
  "hard_constraints": {
    "max_price": 80.0,
    "min_rating": null,
    "must_include_terms": [
      "headphones"
    ],
    "use_case": null,
    "cheaper_than_reference": false,
    "same_source_as_reference": false
  },
  "soft_preferences": [
    "highly_rated"
  ],
  "reference_items": [],
  "excluded_brands": [
    "Beats"
  ],
  "excluded_terms": [],
  "grounding_needs": [
    "price",
    "rating",
    "brand_constraint"
  ],
  "clarification_needed": false,
  "clarification_questions": []
}
reference_cheaper_headphones
I want something like this item but cheaper.
{
  "original_query": "I want something like this item but cheaper.",
  "user_intent": "similar_i

## Hand-Off To The Next Stages

The point of this notebook is not just to emit a parsed object. It is to emit the *right* parsed object: one that retrieval, reranking, grounding, and response generation can consume without inventing new logic or reinterpreting the user request from scratch.


In [12]:
handoff_examples = [
    {
        "label": example["label"],
        "handoff": build_pipeline_handoff(example["parsed"]),
    }
    for example in parsed_examples
]

handoff_df = pd.DataFrame(
    [
        {
            "label": example["label"],
            "candidate_generation_mode": example["handoff"]["candidate_generation_mode"],
            "reference_parent_asin": example["handoff"]["reference_parent_asin"],
            "source_filter": example["handoff"]["source_filter"],
            "retrieval_query": example["handoff"]["retrieval_query"],
            "max_price": example["handoff"]["hard_filters"]["max_price"],
            "excluded_brands": ", ".join(example["handoff"]["hard_filters"]["excluded_brands"]),
            "clarification_needed": example["handoff"]["clarification_needed"],
        }
        for example in handoff_examples
    ]
)

display(handoff_df)

for example in handoff_examples:
    print("=" * 100)
    print(example["label"])
    print(json.dumps(example["handoff"], indent=2, ensure_ascii=False))

,label,candidate_generation_mode,reference_parent_asin,source_filter,retrieval_query,max_price,excluded_brands,clarification_needed
0,budget_headphones_avoid_beats,text_query,NaN,electronics,headphones highly rated,80.00,Beats,False
1,reference_cheaper_headphones,reference_similarity,B08XPWDSWW,electronics,NaN,21.99,,False
2,induction_frying_pan,text_query,NaN,home_and_kitchen,frying pan induction cooking,NaN,,False
3,dumbbells_home_gym,text_query,NaN,sports_and_outdoors,dumbbells dumbbell home gym workouts,NaN,,False
4,lightweight_programming_laptop,text_query,NaN,electronics,laptop programming lightweight,NaN,,False
5,gift_ambiguity,text_query,NaN,NaN,gifting gift,NaN,,True


budget_headphones_avoid_beats
{
  "original_query": "Recommend wireless bluetooth headphones under 80 dollars that are highly rated, but avoid Beats.",
  "candidate_generation_mode": "text_query",
  "reference_parent_asin": null,
  "source_filter": "electronics",
  "same_source_only": false,
  "retrieval_query": "headphones highly rated",
  "hard_filters": {
    "max_price": 80.0,
    "min_rating": null,
    "excluded_brands": [
      "Beats"
    ],
    "excluded_terms": []
  },
  "soft_preferences": [
    "highly_rated"
  ],
  "grounding_needs": [
    "price",
    "rating",
    "brand_constraint"
  ],
  "clarification_needed": false,
  "clarification_questions": []
}
reference_cheaper_headphones
{
  "original_query": "I want something like this item but cheaper.",
  "candidate_generation_mode": "reference_similarity",
  "reference_parent_asin": "B08XPWDSWW",
  "source_filter": "electronics",
  "same_source_only": true,
  "retrieval_query": null,
  "hard_filters": {
    "max_price": 21

## Limits Of The Current Parser

This parser is rule-based, which means it remains brittle around deeper semantics, vague gift-like requests, and indirect reference mentions. At this stage, the project benefits more from a stable and inspectable contract than from a less transparent language model layer.